# Tutorial: Loader and Inventory Smoke Test

Audience:
- AMDockVS developers.

Prerequisites:
- AMDockVS, MolSuite, and the executors are available in the environment.

Learning goals:
- Verify the basic import path through loader.
- Check the basic ligand and receptor inventory.

## Outline

1. Set up
2. Create a temporary project
3. Import a sample ligand and receptor
4. Inspect the inventory and statistics

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="02_loader_inventory_", dir=TMP_ROOT))
LIGAND_FILE = REPO_ROOT / "data" / "docking" / "1iep_ligand.sdf"
RECEPTOR_FILE = REPO_ROOT / "data" / "docking" / "1iep_receptor.pdbqt"
assert LIGAND_FILE.exists(), LIGAND_FILE
assert RECEPTOR_FILE.exists(), RECEPTOR_FILE

runtime = AMDockVSRuntime()
runtime.create_project(
    name=f"loader-inventory-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for loader and inventory.",
)

{
    "ligand_file": str(LIGAND_FILE),
    "receptor_file": str(RECEPTOR_FILE),
    "project_dir": str(PROJECT_DIR),
}


In [ ]:
job_ids = [
    *runtime.loader.load_ligands([LIGAND_FILE], executor_name="process"),
    *runtime.loader.load_receptors([RECEPTOR_FILE], executor_name="process"),
]
status_map = runtime.wait_for_jobs(job_ids, timeout_s=180.0, poll_s=0.2)
assert all(row.status == "completed" for row in status_map.values()), status_map
status_map


In [ ]:
from amdockvs.api_common import project_queries

ligand_scope = runtime.molecules.select(role="ligand")
receptor_scope = runtime.molecules.select(role="receptor")

ligand_stats = project_queries(runtime).get_ligand_table_stats()
receptor_stats = project_queries(runtime).get_receptor_table_stats()

ligands = list(runtime.molecules.stream(ligand_scope))
receptors = list(runtime.molecules.stream(receptor_scope))

assert ligand_stats.total_ligands >= 1
assert receptor_stats.total_receptors >= 1
assert ligands, "No ligands imported"
assert receptors, "No receptors imported"

{
    "ligand_count": ligand_stats.total_ligands,
    "receptor_count": receptor_stats.total_receptors,
    "first_ligand": {
        "id": ligands[0].id,
        "name": ligands[0].name,
        "format": ligands[0].input_format,
    },
    "first_receptor": {
        "id": receptors[0].id,
        "name": receptors[0].name,
        "format": receptors[0].input_format,
    },
}
